In [ ]:
import matplotlib.pyplot as plt
import numpy as np

x = np.linspace(0, 2, 100)

plt.plot(x, x) # Plot some data on the (implicit) axes. (x, y)
plt.plot(x, x**2)
plt.plot(x, x**3)

plt.xlabel('x label')
plt.ylabel('y label')

plt.title("Simple Plot")
# plt.xlim(0,2)
# plt.ylim(0,2)

plt.legend(['linear', 'quadratic', 'cubic']);



In [ ]:
import matplotlib.pyplot as plt
import numpy as np

x = np.linspace(0, 2, 100) # Sample data. 

# Note that even in the OO-style, we use `.pyplot.figure` to create the Figure. 
fig, ax = plt.subplots()

ax.plot(x, x) # Plot some data on the axes. 
ax.plot(x, x**2) # Plot more data on the axes... 
ax.plot(x, x**3) # ... and some more. 
ax.set_xlabel('x label') # Add an x-label to the axes. 
ax.set_ylabel('y label') # Add a y-label to the axes. 
ax.set_title("Simple Plot") # Add a title to the axes. 
ax.legend(['linear', 'quadratic', 'cubic']); # Add a legend. 

# ax.set_xlim(0, 1)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

x = np.linspace(0, 2, 100) # Sample data.

ax1 = plt.subplot(121)
ax2 = plt.subplot(122)

ax1.plot(x, x**2)
ax2.plot(x, x**3)

ax1.set_xlabel('x label')
ax1.set_ylabel('y label')

ax2.set_xlabel('x label')

ax1.set_xlim(0, 1.0)
ax2.set_xlim(0, 2.0)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os 

images_path = r'../../images'

images_list = os.listdir(images_path)

image = plt.imread(os.path.join(images_path, images_list[0]))
ax = plt.imshow(image)

plt.axis('off')
plt.title('Image')
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt 

#We calculated the mean, which is the average measurement. Clinician 1’s average was 6.35 cm, and Clinician 2’s was 5.94 cm.
#We also calculated the standard deviation, which tells us how spread out the measurements are. It was about 2.4 cm for both 
# clinicians, so their measurements had a similar amount of variation.
#To find the most common measurement range, we grouped the numbers into 1 cm bins, such as 6 to less than 7 cm. Across both 
# files, the busiest bin was 7 to less than 8 cm.

c1 = np.loadtxt("./clinician_1.txt")
c2 = np.loadtxt("./clinician_2.txt")
assert c1.shape == c2.shape

In [ ]:
print(c1.min(), c2.min(), c1.max(), c2.max())

In [ ]:
# Fixed 1 cm bins: [2, 3), [3, 4), ...

bins = np.arange(2, 16, 1)
print(bins)

for name, values in [("Clinician 1", c1), ("Clinician 2", c2)]:
    counts, edges = np.histogram(values, bins=bins)
    modal = np.flatnonzero(counts == counts.max())
    print(name)
    print("Mean:", np.mean(values))
    print("Sample SD:", np.std(values, ddof=1))
    print("Median:", np.median(values))
    print("Range:", np.min(values), np.max(values))
    print("Modal bins:", [(edges[i], edges[i + 1]) for i in modal])
    print("Modal bin count:", counts.shape, counts.max())

Say scores run from 0 to 100 and you choose 5 bins:

edges:   0      20      40      60      80      100
         |------|-------|-------|-------|-------|

bins:     bin 1   bin 2   bin 3   bin 4   bin 5
         (0–20) (20–40) (40–60) (60–80) (80–100)

bins = 5 means “split the range into 5 buckets”.

edges = [0, 20, 40, 60, 80, 100], the 6 lines that separate them. There is always one more edge than bins, just as 5 fence sections need 6 posts.

counts = how many students landed in each bucket, e.g. [2, 5, 12, 8, 3].


In [ ]:
# Calculate paired differences 
difference = c2 - c1
absolute_difference = np.abs(difference)
average = (c1 + c2) / 2


print("Mean absolute disagreement:", absolute_difference.mean())
print("Mean signed difference:", difference.mean())

In [ ]:
flagged_mask = (absolute_difference > 1)
flagged = np.flatnonzero(flagged_mask) # True is 1, False is 0, so we can use np.flatnonzero to get the indices of flagged cases
print("Case, clinician 1, clinician 2, absolute disagreement:")
print(np.column_stack((
    flagged + 1, c1[flagged], c2[flagged], absolute_difference[flagged]
)))

fig = plt.figure(figsize=(10, 5))

# 1. Histogram: compare measurement distributions
bins = np.arange(
    np.floor(min(c1.min(), c2.min())),
    np.ceil(max(c1.max(), c2.max())) + 1,
    1
)

plt.hist(
    [c1, c2],
    bins=bins,
    label=["Clinician 1", "Clinician 2"],
    color=["steelblue", "orange"]
)
plt.title("Measurement distributions")
plt.xlabel("Measurement (cm)")
plt.ylabel("Number of measurements")
plt.legend()
plt.show()


In [ ]:
fig, axes = plt.subplots(1,2)
axes[0].hist(c1, bins=bins, color=["yellow"]) 

axes[0].set(
    title="Measurement distributions C1",
    xlabel="Measurement (cm)",
    ylabel="Number of measurements",
)

axes[1].hist(c2, bins=bins, color="red")
axes[1].set(
    title="Measurement distributions C2",
    xlabel="Measurement (cm)",
    ylabel="Number of measurements"
)

# plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import gaussian_kde
from scipy.stats import norm


fig, axes = plt.subplots(1,2)
counts, edges, _ = axes[0].hist(c1, bins=bins, color="gold", rwidth=0.8)
kde = gaussian_kde(c1)
x = np.linspace(edges[0], edges[-1], 300)
bin_width = edges[1] - edges[0]

axes[0].plot(x, kde(x) * len(c1) * bin_width, color="black", lw=2, label="trend (KDE)")
axes[0].set(
    title="Measurement distributions C1",
    xlabel="Measurement (cm)",
    ylabel="Number of measurements",
)


mu, sigma = np.mean(c2), np.std(c2)
axes[1].hist(c2, bins=bins, color="red", rwidth=0.8)
axes[1].plot(x, norm.pdf(x, mu, sigma) * len(c2) * bin_width,
             color="blue", lw=2, ls="--", label=f"normal fit (μ={mu:.1f}, σ={sigma:.1f})")
axes[1].set(
    title="Measurement distributions C2",
    xlabel="Measurement (cm)",
    ylabel="Number of measurements"
)

# plt.tight_layout()
plt.show()

In [ ]:
flagged = np.abs(difference) > 1 # flag differences >1 between two clinicians 
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

axes[0].hist(
    [c1, c2],
    bins=bins,
    label=["Clinician 1", "Clinician 2"],
    color=["steelblue", "orange"]
)
axes[0].set(
    title="Measurement distributions",
    xlabel="Measurement (cm)",
    ylabel="Number of measurements"
)
axes[0].legend()

# 2. Scatter plot: compare paired measurements
minimum = min(c1.min(), c2.min()) - 0.5
maximum = max(c1.max(), c2.max()) + 0.5

axes[1].plot(
    [minimum, maximum], [minimum, maximum],
    "k--", label="Equal measurements"
)
axes[1].scatter(
    c1[~flagged_mask], c2[~flagged_mask],
    color="steelblue", label="Difference ≤1 cm"
)
axes[1].scatter(
    c1[flagged_mask], c2[flagged_mask],
    color="crimson", label="Difference >1 cm"
)
axes[1].set(
    title="Paired measurements",
    xlabel="Clinician 1 (cm)",
    ylabel="Clinician 2 (cm)",
    xlim=(minimum, maximum),
    ylim=(minimum, maximum)
)
axes[1].set_aspect("equal", adjustable="box")
axes[1].legend()


bias = np.mean(difference)
sd_difference = np.std(difference, ddof=1)
lower_limit = bias - 1.96 * sd_difference
upper_limit = bias + 1.96 * sd_difference

# 3. Bland–Altman plot: differences against average size
axes[2].axhspan(-1, 1, color="lightgrey", alpha=0.4,
                label="Within ±1 cm")
axes[2].scatter(
    average[~flagged_mask], difference[~flagged_mask],
    color="steelblue"
)
axes[2].scatter(
    average[flagged_mask], difference[flagged_mask],
    color="crimson"
)

axes[2].axhline(
    bias, color="steelblue",
    label=f"Mean difference: {bias:.3f} cm"
)
axes[2].axhline(
    lower_limit, color="black", linestyle="--",
    label="Mean difference ±1.96 SD"
)
axes[2].axhline(upper_limit, color="black", linestyle="--")

for i in np.flatnonzero(flagged):
    axes[2].annotate(
        str(i + 1),
        (average[i], difference[i]),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=8
    )

axes[2].set(
    title="Bland–Altman agreement",
    xlabel="Mean of the two measurements (cm)",
    ylabel="Clinician 2 − Clinician 1 (cm)"
)
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig("clinician_visualisation.png", dpi=200)
plt.show()